# Bangla Hallucination Detection — Starter Notebook

Predict whether a Bangla LLM response (`response_bn`) to a prompt (`prompt_bn`), optionally
grounded in a `context`, is faithful (`label = 1`) or hallucinated (`label = 0`).

Full column/format details are in `dataset_description.md`.

**Outline**
1. Load the data
2. Explore it (label balance, context coverage, text lengths)
3. Build a TF-IDF + Logistic Regression baseline
4. Evaluate it with cross-validation
5. Produce a `submission.csv`


## 1. Load the data

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
)

RANDOM_STATE = 42
DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"


In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"{len(df)} rows, columns: {list(df.columns)}")
df.head()


In [ ]:
# A handful of responses are purely numeric ("-2", "3", ...) and load from JSON as
# ints rather than strings, which breaks any text vectorizer downstream. Coerce both
# text fields to str so every row is treated as text.
for col in ["prompt_bn", "response_bn"]:
    df[col] = df[col].astype(str)

# The context column uses several stand-ins for "no context" (missing value, empty
# string, the literal string "[NULL]"). Normalize all of them to "" and derive a flag.
NO_CONTEXT_VALUES = {"", "nan", "NaN", "[NULL]", None}

def clean_context(value):
    if pd.isna(value) or str(value).strip() in NO_CONTEXT_VALUES:
        return ""
    return str(value).strip()

df["context"] = df["context"].apply(clean_context)
df["has_context"] = df["context"].str.len() > 0


## 2. Explore the data

In [ ]:
print("Label balance (1 = faithful, 0 = hallucinated):")
print(df["label"].value_counts(normalize=True).rename("share").to_frame())

print("\nContext coverage (rows with vs. without a supporting passage):")
print(df["has_context"].value_counts())

print("\nLabel balance split by whether context is present:")
print(df.groupby("has_context")["label"].value_counts(normalize=True).unstack().round(3))


In [ ]:
text_cols = ["prompt_bn", "response_bn", "context"]
for col in text_cols:
    df[f"{col}_len"] = df[col].str.len()

fig, axes = plt.subplots(1, len(text_cols), figsize=(14, 3.5))
for ax, col in zip(axes, text_cols):
    for label, subset in df.groupby("label"):
        ax.hist(subset[f"{col}_len"], bins=30, alpha=0.5, label=f"label={label}")
    ax.set_title(f"{col} length")
    ax.legend()
plt.tight_layout()
plt.show()


## 3. Baseline model

A text-classification baseline: character n-gram TF-IDF on `prompt_bn` and `response_bn`
separately (character n-grams tend to work better than word n-grams for Bangla, since word
tokenization is less reliable), concatenated and fed into a Logistic Regression.

This ignores `context` entirely — see the ideas at the bottom of this notebook for how to
bring it in.


In [ ]:
def select_column(name):
    # .to_numpy() (rather than returning the raw Series) sidesteps a pandas-output
    # auto-wrapping quirk in recent scikit-learn versions that otherwise corrupts the text.
    return FunctionTransformer(lambda frame: frame[name].to_numpy(), validate=False)

def tfidf_branch(name):
    return Pipeline([
        ("select", select_column(name)),
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=20_000)),
    ])

baseline = Pipeline([
    ("features", FeatureUnion([
        ("prompt", tfidf_branch("prompt_bn")),
        ("response", tfidf_branch("response_bn")),
    ])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])


## 4. Cross-validated evaluation

In [ ]:
def report(y_true, y_pred, y_score=None, name="model"):
    print(f"{'=' * 45}\n  {name}\n{'=' * 45}")
    print(f"  Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Macro-F1   : {f1_score(y_true, y_pred, average='macro'):.4f}  <- primary metric")
    print(f"  F1 (label=1): {f1_score(y_true, y_pred, pos_label=1):.4f}")
    print(f"  F1 (label=0): {f1_score(y_true, y_pred, pos_label=0):.4f}")
    if y_score is not None:
        print(f"  AUC-ROC    : {roc_auc_score(y_true, y_score):.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["hallucinated (0)", "faithful (1)"]))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y = df["label"].values

cv_pred = cross_val_predict(baseline, df, y, cv=cv, method="predict")
cv_proba = cross_val_predict(baseline, df, y, cv=cv, method="predict_proba")[:, 1]

report(y, cv_pred, cv_proba, name="TF-IDF + Logistic Regression (5-fold CV)")


In [ ]:
print("Confusion matrix (rows = true, cols = predicted), order = [0, 1]:")
print(confusion_matrix(y, cv_pred))


## 5. Fit on all data and generate a submission

Once a real `test.csv` is available (same columns, minus `label`), point `TEST_PATH` at it
and run this cell to produce `submission.csv` in the format described in
`dataset_description.md` (`id`, `label`).


In [ ]:
baseline.fit(df, y)

TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"  # update once the real test file is available

try:
    test_df = pd.read_csv(TEST_PATH)
    for col in ["prompt_bn", "response_bn"]:
        test_df[col] = test_df[col].astype(str)
    test_df["context"] = test_df["context"].apply(clean_context)
    test_pred = baseline.predict(test_df)

    submission = pd.DataFrame({
        "id": test_df["id"] if "id" in test_df.columns else range(len(test_df)),
        "label": test_pred,
    })
    submission.to_csv("submission.csv", index=False)
    print(f"Wrote submission.csv with {len(submission)} rows.")
    submission.head()
except FileNotFoundError:
    print(f'No test file found at "{TEST_PATH}" yet — this cell will work once one is added.')


## Ideas to improve on this baseline

- **Bring in `context`**: add features such as token/n-gram overlap, or the NLI entailment
  score from section 6, between `context` and `response_bn` — a faithful answer should be
  grounded in the context when one is given, and closed-book rows (`has_context == False`)
  likely need a different strategy (fact-checking rather than grounding).
- **Stack the NLI score into the baseline**: append `nli_entailment` as an extra feature
  column (with e.g. `0.5` or the has_context mean as a fill value for closed-book rows)
  before the `LogisticRegression` step, rather than only evaluating it standalone.
- **Try multilingual embeddings**: a sentence-transformer that supports Bangla, feeding a
  shallow classifier, in place of char-TF-IDF.
- **Error-analyze by subset**: this dataset mixes closed-book trivia, open-book QA, and
  proverb interpretation — check that macro-F1 holds up across all of them, not just in
  aggregate.


In [ ]:
import torch
from transformers import pipeline

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
print(f'Loading NLI model: {NLI_MODEL}')
nli_pipe = pipeline(
    'zero-shot-classification',
    model=NLI_MODEL,
    device=0 if device == 'cuda' else -1,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
print('NLI model ready.')


In [ ]:
from tqdm.auto import tqdm

def nli_entailment_score(context, response):
    result = nli_pipe(
        context,
        candidate_labels=[response],
        hypothesis_template="{}",
        multi_label=True,
    )
    return result["scores"][0]

context_rows = df[df["has_context"]]
scores = [
    nli_entailment_score(row.context, row.response_bn)
    for row in tqdm(context_rows.itertuples(), total=len(context_rows))
]

df["nli_entailment"] = np.nan
df.loc[context_rows.index, "nli_entailment"] = scores


In [ ]:
# Sanity-check the signal on its own: threshold it at 0.5 and score against the true
# labels, restricted to the context rows it actually applies to.
context_eval = df.loc[df["has_context"]].dropna(subset=["nli_entailment"])
nli_pred = (context_eval["nli_entailment"] > 0.5).astype(int)

report(
    context_eval["label"], nli_pred, context_eval["nli_entailment"],
    name="NLI entailment @ 0.5 threshold (context rows only)",
)


## Ideas to improve on this baseline

- **Bring in `context`**: add features such as token/n-gram overlap between `context` and
  `response_bn` — a faithful answer should be grounded in the context when one is given, and
  closed-book rows (`has_context == False`) likely need a different strategy (fact-checking
  rather than grounding).
- **Try multilingual embeddings**: a sentence-transformer that supports Bangla, feeding a
  shallow classifier, in place of char-TF-IDF.
- **Error-analyze by subset**: this dataset mixes closed-book trivia, open-book QA, and
  proverb interpretation — check that macro-F1 holds up across all of them, not just in
  aggregate.
